In [1]:
import pandas as pd
import re
from pathlib import Path
from collections import Counter

pd.set_option("display.max_colwidth", 120)

DATA_DIR = Path("../data")
RAW_PATH = DATA_DIR / "raw/lyrics_raw.csv"
OUT_PATH = DATA_DIR / "processed/lyrics_clean.csv"

In [2]:
df = pd.read_csv(RAW_PATH)

df.columns

Index(['artist', 'song_title', 'album', 'release_year', 'lyrics', 'source'], dtype='object')

In [3]:
df[[c for c in df.columns if c != "lyrics"][:3] + ["lyrics"]].sample(5, random_state=7)

,artist,song_title,album,lyrics
27,The O.C. Supertones,Supertones Strike Back,"{'api_path': '/albums/232434', 'cover_art_url': 'https://images.genius.com/3f648b3026b13058a78f028804281965.600x600x...","Supertones strike back\nJust like Leia's Father\nYou hit, we hit back harder\nLike Huss and Stephen, I am not afraid..."
2,Goldfinger,Here in Your Bedroom,"{'api_path': '/albums/1098558', 'cover_art_url': 'https://images.genius.com/b4a22fef1f73ee5a3bd2b9081b807b8c.999x999...",Here in your bedroom\nI can turn my head off\nThe less that I feel\nIs the less that I'm on top\nI wonder what you t...
41,The Hippos,Far Behind,"{'api_path': '/albums/171437', 'cover_art_url': 'https://images.genius.com/a736551c0ede74ff768b93c9996dac15.1000x100...",All I need is a walking cane\nTo guide me on my way\n\nAll I need is a walking cane\nTo guide me on my way\nA hat to...
56,Choking Victim,500 Channels,"{'api_path': '/albums/263205', 'cover_art_url': 'https://images.genius.com/0253f533c4a48c0746e2dd90f662b4b7.500x500x...",Five hundred channels of a daydream stimulation\nHelps me to resent my life and raise my expectations\nLocked into r...
18,Sublime,Santeria,"{'api_path': '/albums/547549', 'cover_art_url': 'https://images.genius.com/888a662c5f1ed76bbad99cae2960c4ab.640x640x...","I don't practice Santeria, I ain't got no crystal ball\nWell, I had a million dollars, but I'd, I'd spend it all\nIf..."


In [4]:
### Cleaning ###
BRACKET_TAG_REGEX = re.compile(r"\[(chorus|verse|bridge|intro|outro|pre[- ]chorus|hook|refrain).*?\]", re.IGNORECASE)

def clean_lyrics(text: str) -> str:
    if pd.isna(text):
        return ""
    
    # normalize newlines
    t = str(text).replace("\r\n", "\n").replace("\r", "\n")

    # remove common section labels, like: [Chorus], [Verse 2], etc...
    t = BRACKET_TAG_REGEX.sub("", t)

    # lowercasse
    t = t.lower()

    # remove URLs if they exist
    t = re.sub(r"https?://\S+|www\.\S+", "", t)

    # keep letters + spaces + newlines; drop punctuation/numbers
    t = re.sub(r"[^a-z\s\n']", " ", t)  # keep apostrophes for don't, i'm

    # normalize spaces within lines
    t = re.sub(r"[ \t]+", " ", t)

    # collapse 3+ blank lines to 2 (keeps verse breaks)
    t = re.sub(r"\n{3,}", "\n\n", t)

    # strip whitespace on each line
    t = "\n".join(line.strip() for line in t.split("\n"))

    # remove leading/trailing blank lines
    t = t.strip()

    return t

In [5]:
df["lyrics_raw"] = df["lyrics"].astype(str)
df["lyrics_clean"] =df["lyrics_raw"].apply(clean_lyrics)

# Show a few before and after samples
sample = df.sample(3, random_state=3)[["artist", "title", "lyrics_raw", "lyrics_clean"]] if "artist" in df.columns and "title" in df.columns else df.sample(3, random_state=3)[["lyrics_raw", "lyrics_clean"]]
sample

,lyrics_raw,lyrics_clean
6,"Annie's twelve years old, in two more, she'll be a whore\nNobody ever told her it's the wrong way\nDon't be afraid w...",annie's twelve years old in two more she'll be a whore\nnobody ever told her it's the wrong way\ndon't be afraid wit...
23,"It was the summer of 95 (so what!)\nIn the backyard, shaving the old plies\nFeeling so strong (strong!)\nSomething w...",it was the summer of so what\nin the backyard shaving the old plies\nfeeling so strong strong\nsomething went wrong ...
13,"Just talked to this girl used to live here on my street\nAfter all these years you're here, and you remember me\nShe...",just talked to this girl used to live here on my street\nafter all these years you're here and you remember me\nshe ...


In [6]:
# Sanity Checks

def count_lines(text: str) -> int:
    return sum(1 for l in text.split("\n") if l.strip())

def word_count(text: str) -> int:
    return len([w for w in text.split() if w.strip()])

checks = pd.DataFrame({
    "lines_raw": df["lyrics_raw"].apply(count_lines),
    "lines_clean": df["lyrics_clean"].apply(count_lines),
    "word_raw": df["lyrics_raw"].apply(word_count),
    "words_clean": df["lyrics_clean"].apply(word_count),
})

checks.describe()

,lines_raw,lines_clean,word_raw,words_clean
count,65.000000,65.000000,65.000000,65.000000
mean,43.030769,43.030769,288.015385,290.153846
std,15.352086,15.352086,125.075913,125.311142
min,14.000000,14.000000,144.000000,144.000000
25%,33.000000,33.000000,214.000000,216.000000
50%,41.000000,41.000000,261.000000,270.000000
75%,48.000000,48.000000,330.000000,336.000000
max,96.000000,96.000000,1046.000000,1052.000000


In [7]:
# Quick top words check
all_words = " ".join(df["lyrics_clean"]).split()
counts = Counter(all_words)

counts.most_common(30)

[('i', 729),
 ('the', 610),
 ('you', 490),
 ('and', 454),
 ('to', 445),
 ('a', 380),
 ('it', 291),
 ('my', 285),
 ('me', 252),
 ('that', 232),
 ("i'm", 197),
 ('in', 197),
 ('on', 176),
 ('all', 157),
 ('so', 151),
 ('of', 142),
 ('but', 141),
 ("don't", 135),
 ('know', 133),
 ('your', 133),
 ('be', 130),
 ("it's", 128),
 ('no', 126),
 ('time', 120),
 ('oh', 119),
 ('just', 113),
 ('is', 113),
 ('got', 99),
 ('never', 97),
 ('out', 95)]

In [8]:
DATA_DIR.mkdir(exist_ok=True)

# Keep original metadata + both lyrics columns
df.to_csv(OUT_PATH, index=False)
OUT_PATH

PosixPath('../data/processed/lyrics_clean.csv')